# SQL Analysis in R

Loads cleaned files into an in-memory SQLite database and answers the business questions from the case study.

## 1. Setup

In [ ]:
install.packages(c('DBI', 'RSQLite', 'dplyr', 'ggplot2', 'knitr', 'scales'),
                 repos = 'https://cloud.r-project.org', quiet = TRUE)

library(DBI); library(RSQLite); library(dplyr); library(ggplot2)
library(knitr); library(scales)

options(dplyr.summarise.inform = FALSE)
cat('Packages loaded.\n')

In [ ]:
UPLOAD_DIR <- '/content'

table_map <- c(
  customers  = 'customers_cleaned.csv',
  orders     = 'orders_cleaned.csv',
  deliveries = 'deliveries_cleaned.csv',
  drivers    = 'drivers_cleaned.csv',
  vehicles   = 'vehicles_cleaned.csv',
  hubs       = 'hubs_cleaned.csv',
  complaints = 'complaints_cleaned.csv',
  incidents  = 'incidents_cleaned.csv',
  app_events = 'app_events_cleaned.csv'
)

con <- dbConnect(RSQLite::SQLite(), ':memory:')

for (tbl in names(table_map)) {
  path <- file.path(UPLOAD_DIR, table_map[[tbl]])
  df   <- read.csv(path, stringsAsFactors = FALSE, na.strings = c('', 'NA', 'NULL'))
  dbWriteTable(con, tbl, df, overwrite = TRUE)
  cat(sprintf('  Loaded: %s (%d rows)\n', tbl, nrow(df)))
}

## 2. Query 1 — Failure rate by hub

In [ ]:
q1 <- dbGetQuery(con, "
  SELECT
      h.hub_name AS hub,
      h.hub_id,
      COUNT(*) AS total_deliveries,
      SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed,
      ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*), 1) AS failure_rate_pct,
      CASE
          WHEN 100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*) >= 18 THEN 'High'
          WHEN 100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) / COUNT(*) >= 12 THEN 'Moderate'
          ELSE 'Low'
      END AS risk_level
  FROM deliveries d
  JOIN hubs h ON d.hub_id = h.hub_id
  GROUP BY h.hub_id, h.hub_name
  ORDER BY failure_rate_pct DESC
")
kable(q1, caption = 'Delivery Failure Rates by Hub')

## 3. Query 2 — Driver rating vs training score

In [ ]:
q2 <- dbGetQuery(con, "
  SELECT driver_id, base_zone, training_score, driver_rating
  FROM drivers
  WHERE active_flag = 1 AND training_score IS NOT NULL
  ORDER BY training_score DESC
")

cor_val <- cor(q2$training_score, q2$driver_rating)

ggplot(q2, aes(x = training_score, y = driver_rating)) +
  geom_point(colour = 'steelblue', alpha = 0.6) +
  geom_smooth(method = 'lm', se = FALSE, colour = 'red') +
  labs(title = 'Driver Rating vs Training Score',
       subtitle = paste0('r = ', round(cor_val, 3)),
       x = 'Training Score', y = 'Driver Rating') +
  theme_minimal()

## 4. Query 3 — Complaints by service type and severity

In [ ]:
q3 <- dbGetQuery(con, "
  SELECT
      o.service_type,
      c.severity,
      COUNT(*) AS complaint_count,
      ROUND(AVG(c.resolution_days), 1) AS avg_resolution_days,
      ROUND(SUM(c.compensation_amount), 2) AS total_compensation
  FROM complaints c
  JOIN orders o ON c.order_id = o.order_id
  GROUP BY o.service_type, c.severity
  ORDER BY complaint_count DESC
")
kable(q3, caption = 'Complaints by Service Type and Severity')

In [ ]:
q3_plot <- q3 %>%
  mutate(severity = factor(severity, levels = c('Low', 'Medium', 'High')))

ggplot(q3_plot, aes(x = reorder(service_type, -complaint_count), y = complaint_count, fill = severity)) +
  geom_col() +
  scale_fill_manual(values = c('Low' = '#86EFAC', 'Medium' = '#FCD34D', 'High' = '#F87171')) +
  labs(title = 'Complaints by Service Type and Severity',
       x = 'Service Type', y = 'Complaints', fill = 'Severity') +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 30, hjust = 1))

## 5. Query 4 — Order priority vs revenue and on-time rate

In [ ]:
q4 <- dbGetQuery(con, "
  SELECT
      o.priority_level,
      COUNT(*) AS order_count,
      ROUND(AVG(o.order_value), 2) AS avg_order_value,
      ROUND(SUM(o.order_value), 2) AS total_revenue,
      ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'OnTime' THEN 1 ELSE 0 END)
            / NULLIF(COUNT(d.delivery_id), 0), 1) AS ontime_rate
  FROM orders o
  LEFT JOIN deliveries d ON o.order_id = d.order_id
  GROUP BY o.priority_level
")
kable(q4, caption = 'Order Priority vs Revenue and On-Time Rate')

In [ ]:
q4_plot <- q4 %>%
  mutate(priority_level = factor(priority_level, levels = c('Low', 'Medium', 'High', 'Critical')))

ggplot(q4_plot, aes(x = priority_level, y = avg_order_value, fill = ontime_rate)) +
  geom_col() +
  geom_text(aes(label = paste0(ontime_rate, '% on-time')), vjust = -0.4, size = 3.5) +
  scale_fill_gradient(low = '#FCA5A5', high = '#22C55E', name = 'On-Time %') +
  scale_y_continuous(labels = label_dollar(prefix = '£')) +
  labs(title = 'Order Priority vs Revenue and On-Time Rate',
       x = 'Priority Level', y = 'Average Order Value (£)') +
  theme_minimal()

## 6. Close connection

In [ ]:
dbDisconnect(con)
cat('Connection closed.\n')